<a href="https://colab.research.google.com/github/beingAnujChaudhary/Practical-Statistics-for-DS/blob/main/chapter_01_exploratory_data_analysis/experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

# Mount Google Drive (optional)
drive.mount('/content/drive')

# Clone your GitHub repository
!git clone https://github.com/beingAnujChaudhary/Practical-Statistics-for-DS.git

# Move into repository
%cd /content/Practical-Statistics-for-DS

# Move into Chapter 1 folder
%cd chapter_01_exploratory_data_analysis

Mounted at /content/drive
Cloning into 'Practical-Statistics-for-DS'...
remote: Enumerating objects: 144, done.
remote: Counting objects: 100% (144/144), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 144 (delta 86), reused 72 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (144/144), 504.23 KiB | 11.46 MiB/s, done.
Resolving deltas: 100% (86/86), done.
/content/Practical-Statistics-for-DS
/content/Practical-Statistics-for-DS/chapter_01_exploratory_data_analysis


# Chapter 01 Experiments: Exploratory Data Analysis

> Source: *Practical Statistics for Data Scientists, 2nd Edition*  
> Chapter 1: Exploratory Data Analysis

## Purpose

This notebook is for:
- Experimentation
- Curiosity-driven learning
- Testing assumptions
- Building statistical intuition
- Connecting EDA with machine learning

Unlike `exercises.ipynb`, there are no fixed answers.

The goal is to ask:

> "What happens if I change something?"

---

## 🧪 How to Use This Notebook

- **Experiments ≠ Exercises**: These are open-ended explorations. There are no "right" answers.
- **Modify freely**: Change parameters, swap datasets, break things, and observe.
- **Document insights**: Add your observations in markdown cells as you go.
- **Save interesting outputs**: Export plots to `output/` for your portfolio.

---


## Setup

<details>
<summary>Click to view solution</summary>

```python
# Imports
try:
    from utils.notebook_setup import *
except:
    import warnings
    warnings.filterwarnings("ignore")
    
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    import os
    from scipy.stats import trim_mean, skew
    from scipy import stats
    
    plt.style.use("seaborn-v0_8")
    np.random.seed(42)

print("Experiment notebook ready")
```
</details>


---

## Load Dataset

<details>
<summary>Click to view solution</summary>

```python
# Load state dataset
data_path = '../datasets/raw/state.csv'
if os.path.exists(data_path):
    state = pd.read_csv(data_path)
else:
    url = 'https://raw.githubusercontent.com/gedeck/practical-statistics-for-data-scientists/master/data/state.csv'
    state = pd.read_csv(url)

state.head()
```
</details>


---

# Experiment 1: Mean vs Median Under Outliers

## Question

How sensitive are mean and median to extreme values?

### Hypothesis

Mean will change significantly. Median will remain relatively stable.

## Part A: Artificial Outlier Injection

<details>
<summary>Click to view solution</summary>

```python
state["Population"].describe()
```
</details>

<details>
<summary>Click to view solution</summary>

```python
original_mean = state["Population"].mean()
original_median = state["Population"].median()

print("Original Statistics")
print(f"Mean: {original_mean:,.2f}")
print(f"Median: {original_median:,.2f}")
```
</details>

<details>
<summary>Click to view solution</summary>

```python
# Add artificial outlier
experiment = state.copy()
experiment.loc[len(experiment)] = experiment.iloc[0]
experiment.loc[len(experiment)-1, "Population"] = 500_000_000

new_mean = experiment["Population"].mean()
new_median = experiment["Population"].median()

print("After Extreme Outlier")
print(f"Mean: {new_mean:,.2f}")
print(f"Median: {new_median:,.2f}")
print(f"\nMean changed by: {new_mean - original_mean:,.2f}")
print(f"Median changed by: {new_median - original_median:,.2f}")
```
</details>


## Part B: Systematic Outlier Contamination

### Goal
Quantify how mean, median, and trimmed mean respond to increasing outlier contamination.

<details>
<summary>Click to view solution</summary>

```python
# Generate synthetic "typical" data (roughly normal)
np.random.seed(42)
clean_data = np.random.normal(loc=100, scale=15, size=500)

# Function to add outliers
def add_outliers(data, n_outliers, outlier_magnitude=5):
    """Add extreme values to data"""
    outliers = np.random.choice([1, -1], size=n_outliers) * np.random.exponential(scale=outlier_magnitude * np.std(data), size=n_outliers)
    return np.concatenate([data, data.mean() + outliers])

# Test different contamination levels
contamination_levels = [0, 1, 5, 10, 25, 50]  # number of outliers
results = []

for n_out in contamination_levels:
    contaminated = add_outliers(clean_data, n_out)
    
    results.append({
        'n_outliers': n_out,
        'mean': np.mean(contaminated),
        'median': np.median(contaminated),
        'trimmed_10': trim_mean(contaminated, 0.10),
        'trimmed_20': trim_mean(contaminated, 0.20),
        'std': np.std(contaminated),
        'mad': np.median(np.abs(contaminated - np.median(contaminated)))
    })

results_df = pd.DataFrame(results)
results_df
```
</details>

<details>
<summary>Click to view solution</summary>

```python
# Plot how estimates change with outlier count
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Location estimates
axes[0, 0].plot(results_df['n_outliers'], results_df['mean'], 'o-', label='Mean', linewidth=2)
axes[0, 0].plot(results_df['n_outliers'], results_df['median'], 's--', label='Median', linewidth=2)
axes[0, 0].plot(results_df['n_outliers'], results_df['trimmed_10'], 'd:', label='Trimmed 10%', linewidth=2)
axes[0, 0].axhline(y=np.mean(clean_data), color='gray', linestyle=':', alpha=0.5, label='True Mean')
axes[0, 0].set_xlabel('Number of Outliers')
axes[0, 0].set_ylabel('Estimate Value')
axes[0, 0].set_title('Location Estimates vs Outlier Contamination')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Variability estimates
axes[0, 1].plot(results_df['n_outliers'], results_df['std'], 'o-', label='Std Dev', color='red')
axes[0, 1].plot(results_df['n_outliers'], results_df['mad'], 's--', label='MAD', color='green')
axes[0, 1].axhline(y=np.std(clean_data), color='gray', linestyle=':', alpha=0.5, label='True Std')
axes[0, 1].set_xlabel('Number of Outliers')
axes[0, 1].set_ylabel('Variability Estimate')
axes[0, 1].set_title('Variability Estimates vs Outlier Contamination')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Distribution snapshots
axes[1, 0].hist(clean_data, bins=30, alpha=0.5, label='Clean Data', density=True)
axes[1, 0].hist(add_outliers(clean_data, 25), bins=30, alpha=0.5, label='+25 Outliers', density=True)
axes[1, 0].set_xlabel('Value')
axes[1, 0].set_ylabel('Density')
axes[1, 0].set_title('Effect of Outliers on Distribution Shape')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Relative error plot
true_mean = np.mean(clean_data)
results_df['mean_error_pct'] = np.abs(results_df['mean'] - true_mean) / true_mean * 100
results_df['median_error_pct'] = np.abs(results_df['median'] - true_mean) / true_mean * 100

axes[1, 1].plot(results_df['n_outliers'], results_df['mean_error_pct'], 'o-', label='Mean % Error', linewidth=2)
axes[1, 1].plot(results_df['n_outliers'], results_df['median_error_pct'], 's--', label='Median % Error', linewidth=2)
axes[1, 1].set_xlabel('Number of Outliers')
axes[1, 1].set_ylabel('Percent Error from True Mean')
axes[1, 1].set_title('Estimation Error vs Outlier Count')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
```
</details>


## 🤔 Reflection Prompts

1. Which metric changed more?
2. Why?
3. In which business situations would median be safer?
4. At what contamination level does the mean become "unreliable" for your use case?
5. When might you prefer a 20% trimmed mean over the median?
6. How does MAD compare to standard deviation as outliers increase?
7. **ML Relevance**: If you're building a model on data with potential outliers, which location estimate should inform your feature scaling strategy?


---

# Experiment 2: Distribution Shape

## Question

Is the population distribution normal?

### Hypothesis

Population is likely right-skewed because some states are extremely large.

<details>
<summary>Click to view solution</summary>

```python
plt.figure(figsize=(12, 6))

sns.histplot(state["Population"], bins=20, kde=True)
plt.title("Population Distribution")
plt.show()
```
</details>

<details>
<summary>Click to view solution</summary>

```python
print("Skewness:", state["Population"].skew())
print("Kurtosis:", state["Population"].kurtosis())
```
</details>

## 🤔 Reflection

Interpret the skewness value.

Questions:
1. Is distribution symmetric?
2. Why might population data be skewed?
3. Would mean be trustworthy here?


---

# Experiment 3: Log Transformation

## Question

Can log transformation reduce skewness?

### Why this matters

In machine learning, skewed features often hurt model performance.

<details>
<summary>Click to view solution</summary>

```python
state["LogPopulation"] = np.log1p(state["Population"])

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(state["Population"], kde=True, ax=ax[0])
ax[0].set_title("Original Distribution")

sns.histplot(state["LogPopulation"], kde=True, ax=ax[1])
ax[1].set_title("Log Transformed")

plt.show()

print(f"Original skewness: {state['Population'].skew():.2f}")
print(f"Log-transformed skewness: {state['LogPopulation'].skew():.2f}")
```
</details>

## 🤔 Reflection

Questions:
1. Which distribution looks more normal?
2. Why do ML practitioners use log transforms?
3. When might log transform not be appropriate?


---

# Experiment 4: Histogram Binning Strategies — Do They Change the Story?

### Goal
Test how different binning rules affect the visual interpretation of a distribution.

<details>
<summary>Click to view solution</summary>

```python
# Variable to explore
data = state['Population'] / 1_000_000  # in millions

# Define binning strategies
def get_bins(data, method, n_bins=None):
    """Return bin edges for different strategies"""
    if method == 'sturges':
        n = int(np.ceil(np.log2(len(data)) + 1))
        return np.linspace(data.min(), data.max(), n + 1)
    elif method == 'sqrt':
        n = int(np.sqrt(len(data)))
        return np.linspace(data.min(), data.max(), n + 1)
    elif method == 'fd':  # Freedman-Diaconis
        iqr = np.percentile(data, 75) - np.percentile(data, 25)
        bin_width = 2 * iqr / (len(data) ** (1/3))
        n = int(np.ceil((data.max() - data.min()) / bin_width))
        return np.linspace(data.min(), data.max(), n + 1)
    elif method == 'fixed':
        return np.linspace(data.min(), data.max(), n_bins + 1)
    elif method == 'quantile':
        return np.quantile(data, np.linspace(0, 1, n_bins + 1))
    else:
        return None

methods = ['sturges', 'sqrt', 'fd', 'fixed', 'quantile']
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, method in enumerate(methods):
    if method == 'fixed':
        bins = get_bins(data, method, n_bins=10)
    elif method == 'quantile':
        bins = get_bins(data, method, n_bins=10)
    else:
        bins = get_bins(data, method)
    
    axes[idx].hist(data, bins=bins, edgecolor='black', alpha=0.7)
    axes[idx].axvline(data.mean(), color='red', linestyle='--', label=f'Mean: {data.mean():.2f}')
    axes[idx].axvline(data.median(), color='green', linestyle='--', label=f'Median: {data.median():.2f}')
    axes[idx].set_title(f'{method.upper()} Binning\n{len(bins)-1} bins')
    axes[idx].set_xlabel('Population (millions)')
    axes[idx].set_ylabel('Frequency')
    axes[idx].legend(fontsize=8)
    axes[idx].grid(alpha=0.3)

# Hide unused subplot
axes[-1].axis('off')

plt.suptitle('How Binning Strategy Affects Distribution Interpretation', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()
```
</details>

<details>
<summary>Click to view solution</summary>

```python
# Compute summary stats under each binning
binning_stats = []

for method in methods:
    if method in ['fixed', 'quantile']:
        bins = get_bins(data, method, n_bins=10)
    else:
        bins = get_bins(data, method)
    
    # Count per bin
    counts, _ = np.histogram(data, bins=bins)
    
    binning_stats.append({
        'method': method,
        'n_bins': len(bins) - 1,
        'min_bin_count': counts.min(),
        'max_bin_count': counts.max(),
        'empty_bins': np.sum(counts == 0),
        'skew_visual': 'Right' if np.mean(data) > np.median(data) else 'Left/Symmetric'
    })

pd.DataFrame(binning_stats)
```
</details>

### 🤔 Reflection Prompts
1. Which binning method best reveals the right-skew of state populations?
2. When might quantile binning be preferable (hint: comparing distributions)?
3. How many empty bins is "too many"? What does that tell you about your data?
4. **ML Relevance**: If you're binning a feature for a tree-based model, does binning strategy matter? What about for logistic regression?



---

# Experiment 5: Missing Data Simulation

## Question

What happens when data is missing?

### Hypothesis

Summary statistics may become unreliable.

<details>
<summary>Click to view solution</summary>

```python
experiment_missing = state.copy()

random_rows = np.random.choice(
    experiment_missing.index,
    size=10,
    replace=False
)

experiment_missing.loc[random_rows, "Population"] = np.nan

print("Missing values:")
print(experiment_missing.isnull().sum())
```
</details>

<details>
<summary>Click to view solution</summary>

```python
print("Original Mean:", state["Population"].mean())
print("Mean with Missing Data:", experiment_missing["Population"].mean())
print("Median with Missing Data:", experiment_missing["Population"].median())
```
</details>

## 🤔 Reflection

Questions:
1. Did the mean change?
2. What happens if missing values are not random?
3. Why does this matter in ML?


---

# Experiment 6: Correlation Exploration

## Question

What happens when two variables are strongly correlated?

<details>
<summary>Click to view solution</summary>

```python
numeric_columns = state.select_dtypes(include=np.number)

correlation = numeric_columns.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation, annot=True, cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.show()
```
</details>

## 🤔 Reflection

Questions:
1. Which variables appear strongly related?
2. Does correlation imply causation?
3. Could hidden variables exist?



---

# Experiment 7: Correlation ≠ Causation — Simulating Confounding

### Goal
Demonstrate how a hidden variable can create spurious correlation.

<details>
<summary>Click to view solution</summary>

```python
np.random.seed(42)
n = 200

# Hidden confounder: "Economic Development"
economic_dev = np.random.normal(0, 1, n)

# Observable variables influenced by confounder
population = 50 + 10 * economic_dev + np.random.normal(0, 2, n)  # richer states grow
murder_rate = 5 - 0.8 * economic_dev + np.random.normal(0, 0.5, n)  # richer states safer

# Create DataFrame
synthetic = pd.DataFrame({
    'Population': population,
    'MurderRate': murder_rate,
    'EconomicDev': economic_dev  # "unobserved" in real analysis
})
```
</details>

<details>
<summary>Click to view solution</summary>

```python
# Compute correlation without knowing about EconomicDev
naive_corr = synthetic['Population'].corr(synthetic['MurderRate'])
print(f"Naive correlation (Population ↔ MurderRate): {naive_corr:.3f}")

# Scatterplot
plt.figure(figsize=(6, 5))
sns.regplot(data=synthetic, x='Population', y='MurderRate', scatter_kws={'alpha':0.6})
plt.title('Naive View: Population vs Murder Rate\n(Ignoring Economic Development)')
plt.xlabel('Population')
plt.ylabel('Murder Rate')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("❌ Misleading conclusion: Larger populations → lower murder rates?")
```
</details>

<details>
<summary>Click to view solution</summary>

```python
# Bin by economic development terciles
synthetic['EconTercile'] = pd.qcut(synthetic['EconomicDev'], q=3, labels=['Low', 'Medium', 'High'])

# Compute correlation within each stratum
stratified_results = []
for level in synthetic['EconTercile'].unique():
    subset = synthetic[synthetic['EconTercile'] == level]
    corr = subset['Population'].corr(subset['MurderRate'])
    stratified_results.append({
        'EconomicLevel': level,
        'n': len(subset),
        'correlation': corr
    })

pd.DataFrame(stratified_results)
```
</details>

<details>
<summary>Click to view solution</summary>

```python
# Visualize: colored by economic development
plt.figure(figsize=(7, 5))
sns.scatterplot(data=synthetic, x='Population', y='MurderRate',
                hue='EconTercile', palette='viridis', s=60, alpha=0.7)
plt.title('Stratified View: Color = Economic Development')
plt.xlabel('Population')
plt.ylabel('Murder Rate')
plt.legend(title='Economic Level')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("✅ Insight: Within similar economic levels, population and murder rate are uncorrelated.")
print("The apparent relationship was driven by the confounder.")
```
</details>

### 🤔 Reflection Prompts
1. How would you detect potential confounders in real-world data?
2. When is stratification feasible vs. when do you need regression adjustment?
3. **ML Relevance**: If you feed the naive correlation into a feature selection algorithm, what could go wrong?
4. Try adding a second confounder. Does the pattern hold?


---

# Experiment 8: Outlier Detection
## Question

Which states are outliers?

<details>
<summary>Click to view solution</summary>

```python
q1 = state["Population"].quantile(0.25)
q3 = state["Population"].quantile(0.75)

iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print(f"Lower bound: {lower_bound:,.0f}")
print(f"Upper bound: {upper_bound:,.0f}")

outliers = state[
    (state["Population"] < lower_bound) |
    (state["Population"] > upper_bound)
]

outliers[['State', 'Population']]
```
</details>

## 🤔 Reflection

Questions:
1. Which states are outliers?
2. Should outliers always be removed?
3. When are outliers actually useful?



---

# Experiment 9: Sampling Effects

## Question

What happens if we only analyse part of the data?

<details>
<summary>Click to view solution</summary>

```python
sample = state.sample(frac=0.5, random_state=42)

print("Full Dataset Mean:", f"{state['Population'].mean():,.0f}")
print("Sample Mean:", f"{sample['Population'].mean():,.0f}")
print("Full Dataset Median:", f"{state['Population'].median():,.0f}")
print("Sample Median:", f"{sample['Population'].median():,.0f}")
```
</details>

<details>
<summary>Click to view solution</summary>

```python
# Try different sample sizes
for frac in [0.1, 0.3, 0.5, 0.7, 0.9]:
    s = state.sample(frac=frac, random_state=42)
    print(f"Sample {frac:.0%}: Mean={s['Population'].mean():,.0f}, Median={s['Population'].median():,.0f}")
```
</details>

## 🤔 Reflection

Questions:
1. Why are the values different?
2. Can a sample represent a population?
3. What happens with smaller samples?

---


# Experiment 10: Sample Size & Estimation Stability

### Goal
Explore how estimate stability improves with sample size using bootstrapping.

<details>
<summary>Click to view solution</summary>

```python
# Use murder rate as our population parameter of interest
full_data = state['Murder.rate'].values
true_mean = np.mean(full_data)
true_median = np.median(full_data)

print(f"Full dataset (n={len(full_data)}): Mean={true_mean:.2f}, Median={true_median:.2f}")
```
</details>

<details>
<summary>Click to view solution</summary>

```python
def bootstrap_stability(data, statistic=np.mean, sample_sizes=[10, 25, 50], n_boot=1000):
    """Assess how estimate variability changes with sample size"""
    results = {}
    
    for n in sample_sizes:
        estimates = []
        for _ in range(n_boot):
            sample = np.random.choice(data, size=n, replace=True)
            estimates.append(statistic(sample))
        
        results[n] = {
            'estimates': estimates,
            'std_error': np.std(estimates),
            'ci_95': np.percentile(estimates, [2.5, 97.5])
        }
    
    return results

# Run for mean and median
sample_sizes = [5, 10, 20, 35, 50]
mean_results = bootstrap_stability(full_data, np.mean, sample_sizes)
median_results = bootstrap_stability(full_data, np.median, sample_sizes)
```
</details>

<details>
<summary>Click to view solution</summary>

```python
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mean estimates
for n in sample_sizes:
    est = mean_results[n]['estimates']
    axes[0].hist(est, bins=20, alpha=0.4, label=f'n={n}', density=True)
axes[0].axvline(true_mean, color='black', linestyle='--', linewidth=2, label='True Mean')
axes[0].set_xlabel('Bootstrap Mean Estimate')
axes[0].set_ylabel('Density')
axes[0].set_title('Sampling Distribution of Mean')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# Median estimates
for n in sample_sizes:
    est = median_results[n]['estimates']
    axes[1].hist(est, bins=20, alpha=0.4, label=f'n={n}', density=True)
axes[1].axvline(true_median, color='black', linestyle='--', linewidth=2, label='True Median')
axes[1].set_xlabel('Bootstrap Median Estimate')
axes[1].set_ylabel('Density')
axes[1].set_title('Sampling Distribution of Median')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
```
</details>

<details>
<summary>Click to view solution</summary>

```python
# Quantify: standard error vs sample size
se_df = pd.DataFrame({
    'sample_size': sample_sizes,
    'mean_se': [mean_results[n]['std_error'] for n in sample_sizes],
    'median_se': [median_results[n]['std_error'] for n in sample_sizes]
})
se_df['theoretical_se_mean'] = np.std(full_data) / np.sqrt(se_df['sample_size'])

se_df
```
</details>

### 🤔 Reflection Prompts
1. Does the median's standard error decrease at the same rate as the mean's (1/√n)?
2. At what sample size do estimates become "stable enough" for your use case?
3. **ML Relevance**: How does this inform train/test split sizes? When is a small validation set risky?
4. Try this with a skewed variable (e.g., Population). Do the patterns change?



---

# Experiment 11: Visualization Choice — When to Use What?
### Goal
Compare boxplot, violin plot, histogram, and ECDF for the same data. Which tells the clearest story?

<details>
<summary>Click to view solution</summary>

```python
# Load data with clear skew
data = state['Population'] / 1_000_000  # millions

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Boxplot
sns.boxplot(y=data, ax=axes[0, 0], color='lightblue')
axes[0, 0].set_ylabel('Population (millions)')
axes[0, 0].set_title('Boxplot\n(Quartiles + Outliers)')
axes[0, 0].grid(axis='y', alpha=0.3)

# 2. Violin Plot (shows density)
sns.violinplot(y=data, ax=axes[0, 1], inner='box', color='skyblue')
axes[0, 1].set_ylabel('Population (millions)')
axes[0, 1].set_title('Violin Plot\n(Density + Quartiles)')
axes[0, 1].grid(axis='y', alpha=0.3)

# 3. Histogram + KDE
sns.histplot(data, bins=15, kde=True, ax=axes[1, 0], color='skyblue', edgecolor='black')
axes[1, 0].set_xlabel('Population (millions)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Histogram + KDE\n(Shape + Modality)')
axes[1, 0].grid(alpha=0.3)

# 4. Empirical CDF
sorted_data = np.sort(data)
ecdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
axes[1, 1].plot(sorted_data, ecdf, linewidth=2)
axes[1, 1].set_xlabel('Population (millions)')
axes[1, 1].set_ylabel('Cumulative Proportion')
axes[1, 1].set_title('Empirical CDF\n(Percentiles Directly)')
axes[1, 1].grid(alpha=0.3)

plt.suptitle('Choosing the Right Visualization for Skewed Data', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()
```
</details>

<details>
<summary>Click to view solution</summary>

```python
# Create a quick reference
visualization_guide = pd.DataFrame({
    'Plot Type': ['Boxplot', 'Violin', 'Histogram+KDE', 'ECDF'],
    'Best For': ['Quick outlier check', 'Shape + outliers', 'Modality + skew', 'Exact percentiles'],
    'Weakness': ['Hides modality', 'Can be busy', 'Bin/KDE choice matters', 'Less intuitive for beginners'],
    'ML Use Case': ['Feature scaling decisions', 'Understanding feature distribution', 'Detecting bimodality for clustering', 'Threshold selection for classification']
})
visualization_guide
```
</details>

### 🤔 Reflection Prompts
1. Which plot made the right-skew most obvious to you? Why?
2. When would you show an ECDF to a non-technical stakeholder vs. a histogram?
3. **ML Relevance**: If you're explaining feature distributions to a model reviewer, which visualization builds the most trust?
4. Try this with a bimodal synthetic dataset. Which plot reveals the two modes best?

---

# ML Relevance

## Why EDA matters before machine learning

Bad EDA can lead to:
- Poor features
- Bad predictions
- Unstable models
- Hidden bias
- Data leakage

Always inspect:
- Distributions
- Outliers
- Missing values
- Skewness
- Correlations



---

# Experiment 12: Your Own Hypothesis — Test It!

### Template for Personal Experiments

<details>
<summary>Click to view solution</summary>

```python
# ### My Hypothesis: [Describe your question]
# Example: "Does log-transforming Population make its distribution more symmetric?"

log_population = np.log1p(state['Population'])  # log(1+x) to handle zeros

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Original
sns.histplot(state['Population'], kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('Original Population')
axes[0].grid(alpha=0.3)

# Log-transformed
sns.histplot(log_population, kde=True, ax=axes[1], color='lightgreen')
axes[1].set_title('Log(Population)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Quantify skewness
print(f"Skewness - Original: {skew(state['Population']):.2f}")
print(f"Skewness - Log-transformed: {skew(log_population):.2f}")

# ### My Insight:
# [Write your conclusion here]
```
</details>

### Prompt Ideas for Your Own Experiments
- What happens if I remove the top 3 most populous states? Does correlation change?
- Can I create a "robust z-score" using median and MAD instead of mean/SD?
- How does subsampling (random 80% of states) affect my estimates?
- What if I group states by region first, then compute statistics?
- Can I detect outliers using isolation forest vs. IQR? Do they agree?

---

# Challenge Experiment (ChatGPT)

Choose one variable and perform:
1. Distribution analysis
2. Outlier analysis
3. Log transformation
4. Missing value simulation
5. Sampling experiment
6. Final interpretation

---

# Personal Observations

## What surprised me?

Write observations here.

---

## Questions I still have

Add doubts here.

---

## Most important lesson from Chapter 1

Write your takeaway here.

---

## 📊 Exporting Your Findings

<details>
<summary>Click to view solution</summary>

```python
# Example: Save the robustness experiment plot
output_dir = 'output'
os.makedirs(output_dir, exist_ok=True)

# Re-run a key visualization with publication styling
plt.figure(figsize=(8, 6))
plt.plot(results_df['n_outliers'], results_df['mean'], 'o-', label='Mean', linewidth=2.5, markersize=8)
plt.plot(results_df['n_outliers'], results_df['median'], 's--', label='Median', linewidth=2.5, markersize=8)
plt.axhline(y=np.mean(clean_data), color='gray', linestyle=':', alpha=0.7, label='True Value')
plt.xlabel('Number of Outliers Added', fontsize=12)
plt.ylabel('Location Estimate', fontsize=12)
plt.title('Robustness of Location Estimates to Outliers', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(alpha=0.4, linestyle='--')
plt.tight_layout()

# Save high-res
plt.savefig(f'{output_dir}/robustness_experiment.png', dpi=300, bbox_inches='tight')
print(f"✅ Saved plot to {output_dir}/robustness_experiment.png")
plt.show()
```
</details>



---

## 📝 Experiment Log

| Date | Experiment | Key Insight | ML Relevance |
|---|---|---|---|
| 2026-05-28 | Outlier robustness | Median stable up to 10% contamination | Use median for scaling features with outliers |
| 2026-05-28 | Binning strategies | FD rule best for skewed data | Consistent binning improves model interpretability |
| 2026-05-28 | Confounding simulation | Spurious correlation from hidden variable | Always consider confounders in feature selection |

*Update this table as you complete experiments.*

---

## Progress Checklist

- [ ] Tested outliers with artificial injection
- [ ] Compared mean vs median under contamination
- [ ] Explored skewness in distributions
- [ ] Tried log transformation
- [ ] Compared histogram binning strategies
- [ ] Simulated missing values
- [ ] Analysed correlations
- [ ] Created confounding simulation
- [ ] Detected outliers with IQR method
- [ ] Investigated sampling effects
- [ ] Ran bootstrap stability experiment
- [ ] Compared visualization types
- [ ] Created personal experiment
- [ ] Reflected on insights
- [ ] Exported key plots

---